In [1]:
import pandas as pd

In [2]:
data = pd.read_csv('/content/drive/MyDrive/bengaluru_house_prices.csv')

In [3]:
data.head()

,area_type,availability,location,size,society,total_sqft,bath,balcony,price
0,Super built-up Area,19-Dec,Electronic City Phase II,2 BHK,Coomee,1056,2.0,1.0,39.07
1,Plot Area,Ready To Move,Chikka Tirupathi,4 Bedroom,Theanmp,2600,5.0,3.0,120.00
2,Built-up Area,Ready To Move,Uttarahalli,3 BHK,NaN,1440,2.0,3.0,62.00
3,Super built-up Area,Ready To Move,Lingadheeranahalli,3 BHK,Soiewre,1521,3.0,1.0,95.00
4,Super built-up Area,Ready To Move,Kothanur,2 BHK,NaN,1200,2.0,1.0,51.00


In [4]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13320 entries, 0 to 13319
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   area_type     13320 non-null  object 
 1   availability  13320 non-null  object 
 2   location      13319 non-null  object 
 3   size          13304 non-null  object 
 4   society       7818 non-null   object 
 5   total_sqft    13320 non-null  object 
 6   bath          13247 non-null  float64
 7   balcony       12711 non-null  float64
 8   price         13320 non-null  float64
dtypes: float64(3), object(6)
memory usage: 936.7+ KB


In [5]:
data.describe()

,bath,balcony,price
count,13247.000000,12711.000000,13320.000000
mean,2.692610,1.584376,112.565627
std,1.341458,0.817263,148.971674
min,1.000000,0.000000,8.000000
25%,2.000000,1.000000,50.000000
50%,2.000000,2.000000,72.000000
75%,3.000000,2.000000,120.000000
max,40.000000,3.000000,3600.000000


In [6]:
data.isnull().sum()

,0
area_type,0
availability,0
location,1
size,16
society,5502
total_sqft,0
bath,73
balcony,609
price,0


In [7]:
data.dropna(subset=['location', 'size', 'bath'], inplace=True)

In [8]:
data.isnull().sum()

,0
area_type,0
availability,0
location,0
size,0
society,5499
total_sqft,0
bath,0
balcony,536
price,0


In [9]:
data['bhk'] = data['size'].apply(lambda x: int(x.split()[0]))

In [10]:
data.drop('size', axis=1, inplace=True)

In [11]:
def availability_check(x):
  if 'Ready To Move' in x:
            return 1
  else:
    return 0

data['availability'] = data['availability'].apply(availability_check)

In [12]:
def convert_sqft(x):
    try:
        if '-' in x:
            nums = x.split('-')
            return (float(nums[0]) + float(nums[1])) / 2
        return float(x)
    except:
        return None

data['total_sqft'] = data['total_sqft'].apply(convert_sqft)

In [13]:
data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 13246 entries, 0 to 13319
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   area_type     13246 non-null  object 
 1   availability  13246 non-null  int64  
 2   location      13246 non-null  object 
 3   society       7747 non-null   object 
 4   total_sqft    13200 non-null  float64
 5   bath          13246 non-null  float64
 6   balcony       12710 non-null  float64
 7   price         13246 non-null  float64
 8   bhk           13246 non-null  int64  
dtypes: float64(4), int64(2), object(3)
memory usage: 1.0+ MB


In [14]:
from sklearn.model_selection import train_test_split

In [15]:
train_set, test_set = train_test_split(data, test_size=0.2, random_state=42)

In [16]:
X_train = train_set.drop('price', axis=1)
y_train = train_set['price']
X_test = test_set.drop('price', axis=1)
y_test = test_set['price']

In [17]:
num_cols = X_train.select_dtypes(include=['int64', 'float64']).columns

In [18]:
num_cols

Index(['availability', 'total_sqft', 'bath', 'balcony', 'bhk'], dtype='object')

In [28]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import OrdinalEncoder
from sklearn.compose import ColumnTransformer

num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy="median")),
    ('std_scaler', StandardScaler())
])

cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy="most_frequent")),
    ('encoder', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))
])

full_pipeline = ColumnTransformer([
    ("num", num_pipeline, ['availability', 'total_sqft', 'bath', 'balcony', 'bhk']),
    ("cat", cat_pipeline, ['area_type', 'location', 'society']),
])


In [29]:
X_train_prepared = full_pipeline.fit_transform(X_train)

In [21]:
X_train_prepared.shape

(10596, 8)

In [22]:
from keras.models import Sequential, Model
from keras.layers import Dense, BatchNormalization, Dropout, Input
from keras.optimizers import Adam, SGD, RMSprop


model = Sequential([
    Input(shape=(8,)),
    Dense(128, activation='relu'),
    Dense(256, activation='relu'),
    Dropout(0.2),
    Dense(128, activation='relu'),
    Dense(1, activation='linear')
])


In [23]:
model.compile(optimizer='adam', loss='mean_squared_error', metrics=['accuracy'])

In [24]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 256)            │         2,304 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 256)            │        65,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 101,121 (395.00 KB)

 Trainable params: 101,121 (395.00 KB)

 Non-trainable params: 0 (0.00 B)

In [25]:
history = model.fit(X_train_prepared, y_train, epochs=20, verbose=1, validation_split=0.2, batch_size=64)

Epoch 1/20
133/133 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - accuracy: 0.0000e+00 - loss: 27550.3574 - val_accuracy: 0.0000e+00 - val_loss: 24054.7773
Epoch 2/20
133/133 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.0000e+00 - loss: 27232.7578 - val_accuracy: 0.0000e+00 - val_loss: 23835.0664
Epoch 3/20
133/133 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.0000e+00 - loss: 27154.4766 - val_accuracy: 0.0000e+00 - val_loss: 23879.6562
Epoch 4/20
133/133 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.0000e+00 - loss: 26870.1660 - val_accuracy: 0.0000e+00 - val_loss: 24282.9727
Epoch 5/20
133/133 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.0000e+00 - loss: 26374.7578 - val_accuracy: 0.0000e+00 - val_loss: 22911.3242
Epoch 6/20
133/133 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.0000e+00 - loss: 25205.2617 - val_accuracy: 0.0000e+00 - val_loss: 20400.1172
Epoch 7/20
133/133 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.0000e+00 - loss: 21681.1523 - val_accuracy: 0.0000e+00 - val_loss: 164

In [30]:
from sklearn.metrics import r2_score
X_test_prepared = full_pipeline.transform(X_test)
y_predKM = model.predict(X_test_prepared)
print('Coefficient of determination of Keras Model')
print(r2_score(y_test,y_predKM))

83/83 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
Coefficient of determination of Keras Model
0.31026689316665657
